# Imbalanced Classes

## Overview

Class imbalance occurs when one class substantially outnumbers another (e.g. 95% negative, 5% positive). Standard classifiers optimise overall accuracy, which they achieve by over-predicting the majority class.

**Strategies (not mutually exclusive):**

| Strategy | Method | When to use |
|---|---|---|
| Algorithm-level | `class_weight="balanced"` | Always try first |
| Oversampling | SMOTE (synthetic minority) | Minority class too sparse |
| Undersampling | Random undersampling | Very large majority class |
| Threshold tuning | Choose threshold on PR curve | When FP/FN costs differ |
| Evaluation | Precision-Recall AUC, F1 | Always -- not just accuracy |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, roc_auc_score,
    precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay, f1_score)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.special import expit

rng = np.random.default_rng(42)
n = 600
elevation  = rng.uniform(50, 400, n)
nitrate    = rng.gamma(2, 2, n)
ph         = rng.normal(7.2, 0.5, n)
# Imbalanced: ~10% positive
log_odds   = -3.5 + 0.003*elevation - 0.15*nitrate + 0.3*ph
label = (expit(log_odds) > 0.5).astype(int)
print(f"Class distribution: {np.bincount(label)} ({label.mean():.3f} positive)")
X = np.column_stack([elevation, nitrate, ph])
X_tr, X_te, y_tr, y_te = train_test_split(X, label, test_size=0.25,
                                            stratify=label, random_state=42)

---
## The Problem: Accuracy is Misleading

In [ ]:
# Baseline: always predict majority class
accuracy_naive = 1 - y_te.mean()
print(f"Naive (all-negative) accuracy: {accuracy_naive:.3f}")
# Standard RF without balancing
rf_std = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_std.fit(X_tr, y_tr)
print(f"\nStandard RF:")
print(classification_report(y_te, rf_std.predict(X_te), target_names=["absent","present"]))
print(f"AUC-ROC: {roc_auc_score(y_te, rf_std.predict_proba(X_te)[:,1]):.3f}")

---
## class_weight="balanced"

In [ ]:
rf_bal = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                 random_state=42, n_jobs=-1)
rf_bal.fit(X_tr, y_tr)
print("Balanced RF:")
print(classification_report(y_te, rf_bal.predict(X_te), target_names=["absent","present"]))
print(f"AUC-ROC: {roc_auc_score(y_te, rf_bal.predict_proba(X_te)[:,1]):.3f}")

---
## Precision-Recall Curve and Threshold Tuning

In [ ]:
probs = rf_bal.predict_proba(X_te)[:,1]
prec, rec, thresholds = precision_recall_curve(y_te, probs)
ap = average_precision_score(y_te, probs)
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(rec, prec, color="#e74c3c", lw=2)
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title(f"Precision-Recall Curve  (AP={ap:.3f})")
axes[0].axhline(y_te.mean(), color="grey", linestyle="--", label="No-skill")
axes[0].legend()
# F1 by threshold
f1_scores = [f1_score(y_te, probs >= t) for t in thresholds]
best_thresh = thresholds[np.argmax(f1_scores)]
axes[1].plot(thresholds, f1_scores, color="steelblue", lw=1.5)
axes[1].axvline(best_thresh, color="#e74c3c", lw=1.5, linestyle="--",
                label=f"Best threshold={best_thresh:.2f}")
axes[1].set_xlabel("Threshold"); axes[1].set_ylabel("F1 score")
axes[1].set_title("F1 Score by Classification Threshold")
axes[1].legend(); plt.tight_layout(); plt.show()
print(f"Best F1 threshold: {best_thresh:.3f}")
print(classification_report(y_te, probs >= best_thresh, target_names=["absent","present"]))

---
## SMOTE Oversampling

In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    sm = SMOTE(random_state=42)
    X_sm, y_sm = sm.fit_resample(X_tr, y_tr)
    print(f"After SMOTE: {np.bincount(y_sm)}")
    rf_sm = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf_sm.fit(X_sm, y_sm)
    print(f"SMOTE RF AUC: {roc_auc_score(y_te, rf_sm.predict_proba(X_te)[:,1]):.3f}")
    print(classification_report(y_te, rf_sm.predict(X_te), target_names=["absent","present"]))
except ImportError:
    print("imbalanced-learn not installed: pip install imbalanced-learn")
    print("SMOTE creates synthetic minority samples by interpolating between nearest neighbours.")

---

## Common Pitfalls

**1. Optimising for accuracy with imbalanced classes**  
A model predicting all-negative achieves 90% accuracy on a 10% positive dataset. Accuracy is meaningless for imbalanced outcomes. Always report precision, recall, F1, and PR-AUC for the minority class.

**2. Applying SMOTE before the train/test split**  
SMOTE generates synthetic points by interpolating between training examples. If applied before splitting, synthetic points derived from test-set neighbours leak test information into training. Always apply SMOTE inside a cross-validation fold or only to training data after splitting.

**3. Using ROC-AUC as the only metric for imbalanced data**  
ROC-AUC is insensitive to class imbalance because it evaluates performance across all possible thresholds including implausible ones. Precision-Recall AUC (average precision) is more informative for imbalanced outcomes.

**4. Not tuning the classification threshold**  
The default 0.5 threshold is rarely optimal for imbalanced classes. Choose the threshold using the Precision-Recall curve based on the relative cost of false positives and false negatives for the specific application.

**5. Over-relying on SMOTE without trying class_weight first**  
`class_weight="balanced"` is zero-cost, requires no resampling, and often matches SMOTE performance. Always try it first before adding the complexity of resampling approaches.
---
*python_methods_library - Samantha McGarrigle*